# Explore slimmer / evaluator masses

Opens slimmed (or evaluated) ROOT files in **coffea** via a fileset built from a
dataset JSON, and plots a few mass distributions with the **hist** package.

Prereqs:
1. Build the dataset JSON (on cmslpc):
   `python scripts/make_dataset_json.py <eos_path> -o datasets.json`
2. Run this notebook from `run3-mj-analyzer/notebooks/`.

Slimmer `events` tree: nested `ScoutingPFJet` record (`.pt/.eta/.phi/.m`), `HT`,
and (MC) `GenJet`. Evaluator output additionally has `SPANetCandidate_mass`
(2 reconstructed gluino candidates per event).

In [ ]:
import sys, pathlib

# Make the package importable without `pip install -e .` (src/ layout).
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import awkward as ak
import hist
import vector
import matplotlib.pyplot as plt
from coffea.nanoevents import NanoEventsFactory, BaseSchema

from run3_mj_analyzer.fileset import load_fileset

vector.register_awkward()  # enables Momentum4D behaviors (.mass, .px, ...)

In [ ]:
JSON_PATH = "../datasets.json"  # written by scripts/make_dataset_json.py

fileset = load_fileset(JSON_PATH)
print("Datasets:")
for name, ds in fileset.items():
    print(f"  {name}: {len(ds['files'])} files")

In [ ]:
# Pick a dataset and open its first file in coffea (eager mode for a notebook).
dataset = next(iter(fileset))
fname, tree = next(iter(fileset[dataset]["files"].items()))
print(f"Opening {dataset}\n  {fname}  (tree: {tree})")

events = NanoEventsFactory.from_root(
    {fname: tree},
    schemaclass=BaseSchema,
    delayed=False,
    metadata={"dataset": dataset},
).events()

events.fields

In [ ]:
def jet_p4(events):
    """Build per-event Momentum4D jets from the slimmer's ScoutingPFJet record.

    Handles both the nested record layout (events.ScoutingPFJet.pt) and a flat
    fallback (events.ScoutingPFJet_pt) depending on how the schema exposes it.
    """
    if "ScoutingPFJet" in events.fields:
        j = events["ScoutingPFJet"]
        pt, eta, phi, m = j.pt, j.eta, j.phi, j.m
    else:
        pt = events["ScoutingPFJet_pt"]
        eta = events["ScoutingPFJet_eta"]
        phi = events["ScoutingPFJet_phi"]
        m = events["ScoutingPFJet_m"]
    return ak.zip(
        {"pt": pt, "eta": eta, "phi": phi, "mass": m},
        with_name="Momentum4D",
    )


jets = jet_p4(events)
njet = ak.num(jets)
print(f"{len(jets)} events, <njet> = {ak.mean(njet):.2f}, min = {ak.min(njet)}")

## 1. Leading-jet mass

In [ ]:
lead = jets[njet >= 1][:, 0]

h_lead = hist.Hist(
    hist.axis.Regular(60, 0, 120, name="m", label="Leading-jet mass [GeV]")
)
h_lead.fill(m=ak.to_numpy(lead.mass))
h_lead.plot(histtype="step")
plt.title(dataset)
plt.show()

## 2. Leading dijet invariant mass

In [ ]:
two = jets[njet >= 2]
dijet = two[:, 0] + two[:, 1]

h_dijet = hist.Hist(
    hist.axis.Regular(60, 0, 2000, name="m", label="Leading dijet mass [GeV]")
)
h_dijet.fill(m=ak.to_numpy(dijet.mass))
h_dijet.plot(histtype="step")
plt.title(dataset)
plt.show()

## 3. Full all-jet system mass

In [ ]:
# Sum the 4-vectors per event component-wise (robust for variable multiplicity).
system = ak.zip(
    {
        "px": ak.sum(jets.px, axis=1),
        "py": ak.sum(jets.py, axis=1),
        "pz": ak.sum(jets.pz, axis=1),
        "E": ak.sum(jets.energy, axis=1),
    },
    with_name="Momentum4D",
)

h_sys = hist.Hist(
    hist.axis.Regular(60, 0, 5000, name="m", label="All-jet system mass [GeV]")
)
h_sys.fill(m=ak.to_numpy(system.mass))
h_sys.plot(histtype="step")
plt.title(dataset)
plt.show()

## 4. SPANet gluino-candidate mass (evaluator output only)

Present only if the file came from run3-mj-evaluator. There are two candidates
per event (the pair-produced gluinos).

In [ ]:
if "SPANetCandidate_mass" in events.fields:
    cand_m = ak.flatten(events["SPANetCandidate_mass"])
    h_cand = hist.Hist(
        hist.axis.Regular(60, 0, 3000, name="m", label="SPANet candidate mass [GeV]")
    )
    h_cand.fill(m=ak.to_numpy(cand_m))
    h_cand.plot(histtype="step")
    plt.title(f"{dataset} - SPANet candidates")
    plt.show()
else:
    print("No SPANetCandidate_mass branch - this is slimmer (pre-evaluator) output.")

## 5. Accumulate over several files

The cells above used one file. To build up statistics, loop over the first few
files of a dataset and fill a single `hist`.

In [ ]:
N_FILES = 5  # bump up (or set to None for all) once you trust it

h = hist.Hist(
    hist.axis.Regular(60, 0, 5000, name="m", label="All-jet system mass [GeV]")
)

items = list(fileset[dataset]["files"].items())[:N_FILES]
for fpath, tname in items:
    ev = NanoEventsFactory.from_root(
        {fpath: tname}, schemaclass=BaseSchema, delayed=False
    ).events()
    j = jet_p4(ev)
    sysm = ak.zip(
        {
            "px": ak.sum(j.px, axis=1),
            "py": ak.sum(j.py, axis=1),
            "pz": ak.sum(j.pz, axis=1),
            "E": ak.sum(j.energy, axis=1),
        },
        with_name="Momentum4D",
    ).mass
    h.fill(m=ak.to_numpy(sysm))

h.plot(histtype="step")
plt.title(f"{dataset} ({len(items)} files)")
plt.show()